# Trip Planner - Part 1: Checkpoints (Memory across turns)

A tiny but real LangGraph project: a **Trip Planner** that keeps track of:
- the list of destinations you want to visit
- your total budget

You will see exactly how `MemorySaver` makes the agent remember from one turn to the next, and how `thread_id` keeps different users' plans separate.

**What is visible in this notebook**
- Without a checkpointer, the plan resets every call. We will print state to prove it.
- With a checkpointer, the plan grows turn by turn. We will print state to prove it.
- Different `thread_id`s = different plans living side-by-side.
- `get_state_history()` lets you walk back through every snapshot.

Models: `gpt-4o-mini`. Secrets are loaded from Colab's Secrets panel.

## 1. Install

In [1]:
%%capture --no-stderr
%pip install -qU langgraph langchain langchain-core langchain-openai pydantic

## 2. Load OPENAI_API_KEY from Colab secrets

In Colab, click the key icon on the left sidebar, add a secret named `OPENAI_API_KEY`, and toggle notebook access on.

In [2]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ.setdefault("LANGSMITH_TRACING", "false")
print("Key loaded:", bool(os.environ.get("OPENAI_API_KEY")))

Key loaded: True


## 3. Define the project: state + planner node

Our state has three fields: `messages` (chat history), `destinations` (list of cities), and `budget` (a number).

Instead of using tools (which add complexity), we use `with_structured_output` to make the LLM return a clean Pydantic object that tells us what to update.

In [3]:
from typing import Annotated, TypedDict, Optional, Literal
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages


class TripState(TypedDict):
    messages: Annotated[list, add_messages]
    destinations: list[str]
    budget: float


class PlannerAction(BaseModel):
    """What the planner should do based on the user's message."""
    action: Literal["add_destination", "set_budget", "show_plan", "other"] = Field(
        description="What kind of update the user is asking for"
    )
    destination: Optional[str] = Field(None, description="City name when action is add_destination")
    budget: Optional[float] = Field(None, description="Budget in USD when action is set_budget")
    reply: str = Field(description="A short friendly reply to the user")


llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
structured_llm = llm.with_structured_output(PlannerAction)


def planner(state: TripState) -> dict:
    current = (
        f"Current destinations: {state.get('destinations', [])}\n"
        f"Current budget: ${state.get('budget', 0)}"
    )
    system = SystemMessage(content=(
        "You are a trip planner. Read the user's latest message and decide what to update.\n"
        + current
    ))

    decision = structured_llm.invoke([system] + state["messages"])

    update = {"messages": [AIMessage(content=decision.reply)]}

    if decision.action == "add_destination" and decision.destination:
        update["destinations"] = state.get("destinations", []) + [decision.destination]
    elif decision.action == "set_budget" and decision.budget is not None:
        update["budget"] = float(decision.budget)

    return update


builder = StateGraph(TripState)
builder.add_node("planner", planner)
builder.add_edge(START, "planner")
builder.add_edge("planner", END)
print("Graph defined.")

Graph defined.


## 4. A helper to print state nicely

We will use this everywhere so the effect of checkpoints is visually obvious.

In [6]:
def show(label, state_or_snapshot):
    if hasattr(state_or_snapshot, "values"):
        values = state_or_snapshot.values
    else:
        values = state_or_snapshot

    # Ensure 'values' is a dictionary before attempting dictionary operations
    if not isinstance(values, dict):
        print(f"Error: Expected a dictionary for state display, but got type: {type(values)}.")
        print(f"Received object: {values}")
        return

    dests = values.get("destinations", [])
    budget = values.get("budget", 0)
    msgs = values.get("messages", [])
    print("-" * 60)
    print(f"  {label}")
    print("-" * 60)
    print(f"  destinations : {dests}")
    print(f"  budget       : ${budget:.2f}")
    if msgs:
        last = msgs[-1]
        kind = type(last).__name__
        print(f"  last message : [{kind}] {last.content[:90]}")
    print()

## 5. DEMO A - No checkpointer = no memory

Watch the state reset on every call. Paris is forgotten the moment we ask the second question.

In [7]:
graph_no_memory = builder.compile()

# Turn 1
out = graph_no_memory.invoke({
    "messages": [HumanMessage(content="I want to go to Paris")],
    "destinations": [],
    "budget": 0,
})
show("After 'I want to go to Paris' (no checkpointer)", out)

# Turn 2 - notice we have to pass full state again, and Paris is GONE
out = graph_no_memory.invoke({
    "messages": [HumanMessage(content="Also add Rome")],
    "destinations": [],
    "budget": 0,
})
show("After 'Also add Rome' (no checkpointer)", out)

print(">>> Paris is missing. Without a checkpointer, every invocation starts fresh. <<<")

Error: Expected a dictionary for state display, but got type: <class 'builtin_function_or_method'>.
Received object: <built-in method values of dict object at 0x7936dbf22080>
Error: Expected a dictionary for state display, but got type: <class 'builtin_function_or_method'>.
Received object: <built-in method values of dict object at 0x7936dbf232c0>
>>> Paris is missing. Without a checkpointer, every invocation starts fresh. <<<


## 6. DEMO B - Add a checkpointer = full memory

Now we compile with `MemorySaver` and pass a `thread_id` in the config. Each turn builds on the previous one. No more passing full state - just the new message.

In [14]:
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage

memory = MemorySaver()

graph = builder.compile(checkpointer=memory)

alice = {
    "configurable": {
        "thread_id": "alice-trip-2026"
    }
}

def show(title, state):
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)

    if isinstance(state, dict):
        for key, value in state.items():
            print(f"\n{key}:")
            print(value)
    else:
        print("Unexpected state type:", type(state))


print("=" * 60)
print("ALICE'S TRIP PLANNING - thread_id = alice-trip-2026")
print("=" * 60)

state = graph.invoke(
    {"messages": [HumanMessage(content="I want to go to Paris")]},
    alice,
)
show("Turn 1", state)

state = graph.invoke(
    {"messages": [HumanMessage(content="Also add Rome")]},
    alice,
)
show("Turn 2", state)

state = graph.invoke(
    {"messages": [HumanMessage(content="Set my budget to 3500 dollars")]},
    alice,
)
show("Turn 3", state)

state = graph.invoke(
    {"messages": [HumanMessage(content="One more - add Barcelona")]},
    alice,
)
show("Turn 4", state)

ALICE'S TRIP PLANNING - thread_id = alice-trip-2026

Turn 1

messages:
[HumanMessage(content='I want to go to Paris', additional_kwargs={}, response_metadata={}, id='220e36f1-e630-499a-8b33-ed19b63ec4d5'), AIMessage(content="Great choice! I've added Paris to your travel plans.", additional_kwargs={}, response_metadata={}, id='ad6831fc-76fa-4125-853d-5e31ef4fe4e9', tool_calls=[], invalid_tool_calls=[])]

destinations:
['Paris']

Turn 2

messages:
[HumanMessage(content='I want to go to Paris', additional_kwargs={}, response_metadata={}, id='220e36f1-e630-499a-8b33-ed19b63ec4d5'), AIMessage(content="Great choice! I've added Paris to your travel plans.", additional_kwargs={}, response_metadata={}, id='ad6831fc-76fa-4125-853d-5e31ef4fe4e9', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Also add Rome', additional_kwargs={}, response_metadata={}, id='4b6cfe81-3124-4685-8050-a9376460a0e3'), AIMessage(content="I've added Rome to your travel plans! Now you have both Paris and Rome

## 7. DEMO C - Different thread_id = different plan

Bob starts a totally separate trip on the same graph instance. His state is isolated from Alice's.

In [20]:
bob = {"configurable": {"thread_id": "bob-trip-2026"}}

state = graph.invoke(
    {"messages": [HumanMessage(content="I'm planning a Japan trip. Add Tokyo.")]},
    bob,
)
show("Bob's plan after turn 1", state)

state = graph.invoke(
    {"messages": [HumanMessage(content="Add Kyoto too")]},
    bob,
)
show("Bob's plan after turn 2", state)

# Alice's plan is untouched
alice_snapshot = graph.get_state(alice)
show(
    "Original alice thread (still the same)",
    graph.get_state(alice).values
)


Bob's plan after turn 1

messages:
[HumanMessage(content="I'm planning a Japan trip. Add Tokyo.", additional_kwargs={}, response_metadata={}, id='d7600fde-45de-4633-ab64-1d676756e5e3'), AIMessage(content="Great choice! I've added Tokyo to your trip plan.", additional_kwargs={}, response_metadata={}, id='8c8f4d4d-5180-4cca-a5fe-e904a730af67', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Add Kyoto too', additional_kwargs={}, response_metadata={}, id='89ac81fa-0d5a-4a6b-b6b3-80e2c7b5635c'), AIMessage(content="I've added Kyoto to your trip plan! You're going to love it.", additional_kwargs={}, response_metadata={}, id='495200fb-0e8c-4479-84f9-8a34a3be1b66', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content="I'm planning a Japan trip. Add Tokyo.", additional_kwargs={}, response_metadata={}, id='4609c672-4d7f-4811-b59f-c1bcd2446520'), AIMessage(content="Tokyo is already in your trip plan! Is there anything else you'd like to add or change?", additional_kwargs={}, r

## 8. DEMO D - Time travel with `get_state_history`

Every super-step gets its own snapshot. We can list them all and inspect each one.

In [17]:
history = list(graph.get_state_history(alice))
print(f"Alice's thread has {len(history)} checkpoints saved.\n")

# History is most-recent first. Reverse it so checkpoint 0 is the earliest.
for i, snap in enumerate(reversed(history)):
    dests = snap.values.get("destinations", [])
    budget = snap.values.get("budget", 0)
    next_nodes = snap.next or ("END",)
    print(f"checkpoint {i:2d} | destinations={dests} | budget=${budget} | next={next_nodes}")

Alice's thread has 12 checkpoints saved.

checkpoint  0 | destinations=[] | budget=$0 | next=('__start__',)
checkpoint  1 | destinations=[] | budget=$0 | next=('planner',)
checkpoint  2 | destinations=['Paris'] | budget=$0 | next=('END',)
checkpoint  3 | destinations=['Paris'] | budget=$0 | next=('__start__',)
checkpoint  4 | destinations=['Paris'] | budget=$0 | next=('planner',)
checkpoint  5 | destinations=['Paris', 'Rome'] | budget=$0 | next=('END',)
checkpoint  6 | destinations=['Paris', 'Rome'] | budget=$0 | next=('__start__',)
checkpoint  7 | destinations=['Paris', 'Rome'] | budget=$0 | next=('planner',)
checkpoint  8 | destinations=['Paris', 'Rome'] | budget=$3500.0 | next=('END',)
checkpoint  9 | destinations=['Paris', 'Rome'] | budget=$3500.0 | next=('__start__',)
checkpoint 10 | destinations=['Paris', 'Rome'] | budget=$3500.0 | next=('planner',)
checkpoint 11 | destinations=['Paris', 'Rome', 'Barcelona'] | budget=$3500.0 | next=('END',)


## 9. DEMO E - Resume from a specific past checkpoint

Pick an earlier snapshot and run from there. Useful for branching ("what if I had said something else?") and debugging.

In [19]:
# Grab a snapshot from earlier in Alice's history
# (history[0] is the most recent; history[-1] is the very first)
earlier = list(graph.get_state_history(alice))[3]
print(f"Resuming from a snapshot where destinations were: {earlier.values.get('destinations')}")
print(f"Snapshot config: {earlier.config}\n")

# Invoking with the earlier snapshot's config forks from that point
forked = graph.invoke(
    {"messages": [HumanMessage(content="Actually, scrap that and add Lisbon instead.")]},
    earlier.config,
)
show("Forked plan after resuming from an earlier snapshot", forked)

# The original alice thread is untouched
show(
    "Original alice thread (still the same)",
    graph.get_state(alice).values
)

Resuming from a snapshot where destinations were: ['Paris', 'Rome', 'Barcelona']
Snapshot config: {'configurable': {'thread_id': 'alice-trip-2026', 'checkpoint_ns': '', 'checkpoint_id': '1f14f79d-ad49-6222-800a-4cbed5d04753'}}


Forked plan after resuming from an earlier snapshot

messages:
[HumanMessage(content='I want to go to Paris', additional_kwargs={}, response_metadata={}, id='220e36f1-e630-499a-8b33-ed19b63ec4d5'), AIMessage(content="Great choice! I've added Paris to your travel plans.", additional_kwargs={}, response_metadata={}, id='ad6831fc-76fa-4125-853d-5e31ef4fe4e9', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Also add Rome', additional_kwargs={}, response_metadata={}, id='4b6cfe81-3124-4685-8050-a9376460a0e3'), AIMessage(content="I've added Rome to your travel plans! Now you have both Paris and Rome on your itinerary.", additional_kwargs={}, response_metadata={}, id='67abb5bd-6adc-44d7-a6ec-b33b9f16ae98', tool_calls=[], invalid_tool_calls=[]), HumanMessa

## 10. Try it yourself - chat with the planner

Run this cell, type messages, type `quit` to stop. Everything you say is saved to the `you` thread, so you can come back and keep planning later (just re-run this cell).

In [28]:
you = {"configurable": {"thread_id": "you-trip"}}

print("Trip planner ready. Type 'quit' to exit, 'show' to print current plan.\n")

while True:
    msg = input("YOU: ").strip()
    if msg.lower() in {"quit", "exit", ""}:
        break
    if msg.lower() == "show":
        show("Current plan", graph.get_state(you).values)
        continue
    result = graph.invoke({"messages": [HumanMessage(content=msg)]}, you)
    print(f"AGENT: {result['messages'][-1].content}")
    print(f"  (destinations now: {result['destinations']}, budget: ${result['budget']})\n")

print("\nFinal plan:")
show(
    "Original alice thread (still the same)",
    graph.get_state(alice).values
)

Trip planner ready. Type 'quit' to exit, 'show' to print current plan.

YOU: blr for 5000
AGENT: I've updated your budget for Bangalore to $5,000!
  (destinations now: ['Bangalore', 'Pune', 'Kolkata'], budget: $5000.0)

YOU: quit

Final plan:

Original alice thread (still the same)

messages:
[HumanMessage(content='I want to go to Paris', additional_kwargs={}, response_metadata={}, id='220e36f1-e630-499a-8b33-ed19b63ec4d5'), AIMessage(content="Great choice! I've added Paris to your travel plans.", additional_kwargs={}, response_metadata={}, id='ad6831fc-76fa-4125-853d-5e31ef4fe4e9', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Also add Rome', additional_kwargs={}, response_metadata={}, id='4b6cfe81-3124-4685-8050-a9376460a0e3'), AIMessage(content="I've added Rome to your travel plans! Now you have both Paris and Rome on your itinerary.", additional_kwargs={}, response_metadata={}, id='67abb5bd-6adc-44d7-a6ec-b33b9f16ae98', tool_calls=[], invalid_tool_calls=[]), HumanMes

## Recap

| What you saw | Code |
|---|---|
| State that resets every call | `builder.compile()` (no checkpointer) |
| State that persists across calls | `builder.compile(checkpointer=MemorySaver())` + `thread_id` in config |
| Isolation between users | Different `thread_id`s |
| Read state without running | `graph.get_state(config)` |
| List all snapshots | `graph.get_state_history(config)` |
| Fork from a past snapshot | `graph.invoke(input, snapshot.config)` |

Next notebook: **Human-in-the-Loop** - the agent pauses and waits for your typed input before committing changes.